In [ ]:
%pip install transformers accelerate bitsandbytes peft trl

In [ ]:
#%hf auth login

In [2]:
!hf download deepseek-ai/deepseek-coder-6.7b-instruct \
  --local-dir /mnt/storage_C1/igorzwirtes/deepseek-coder-6.7b-instruct

Fetching 13 files: 100%|███████████████████████| 13/13 [00:00<00:00, 178.29it/s]
Download complete: : 0.00B [00:00, ?B/s]              ✓ Downloaded
  path: /mnt/storage_C1/igorzwirtes/deepseek-coder-6.7b-instruct
Download complete: : 0.00B [00:00, ?B/s]


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_path = "/mnt/storage_C1/igorzwirtes/deepseek-coder-6.7b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    quantization_config=bnb_config
)

/home/al.igor.zwirtes/Documentos/llm-nl-to-sql/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights:   1%|          | 2/291 [00:04<10:54,  2.27s/it]/home/al.igor.zwirtes/Documentos/llm-nl-to-sql/env/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [01:56<00:00,  2.51it/s]


In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# prepara modelo para 4-bit training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 8,388,608 || all params: 6,748,901,376 || trainable%: 0.1243


In [5]:
def format_example(example):
    return {
        "text": f"""Convert the following natural language question into SQL.
Only output the SQL query.

Question: {example['question']}
SQL: {example['sql']}"""
    }

In [15]:
data = []

with open("dataset/train.csv", "r", encoding="utf-8") as f:
    next(f)  # pula header
    
    for line in f:
        # divide só na PRIMEIRA vírgula
        parts = line.strip().split(",", 1)
        
        if len(parts) != 2:
            continue  # pula linhas quebradas
        
        question, sql = parts
        
        data.append({
            "question": question.strip(),
            "sql": sql.strip().rstrip(";")  # remove ; final
        })

In [16]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

Map: 100%|██████████| 56355/56355 [00:00<00:00, 67813.74 examples/s]


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
    dataset_text_field="text",
)

trainer.train()

NameError: name 'tokenized_dataset' is not defined

In [ ]:
model.save_pretrained("./lora-sql")
tokenizer.save_pretrained("./lora-sql")

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = """Convert the following natural language question into SQL.
Only output the SQL query.

Question: Clients with more than one thousand dollars spent in pushases
SQL:"""

print(pipe(prompt, max_new_tokens=100)[0]["generated_text"])